# Livrable 3 – Génération de légendes d’images (Image captioning)

## Objectif

Ce livrable constitue la dernière étape du pipeline du projet **Leyanda**. L’objectif est de concevoir un modèle de deep learning capable de **générer automatiquement une légende textuelle pour une photographie**, en s'appuyant sur le dataset **MS COCO**.

Le modèle est composé de deux sous-parties :
- Un **réseau de neurones convolutifs (CNN)** pour encoder l’image en une représentation vectorielle.
- Un **réseau de neurones récurrents (RNN)** qui génère une phrase mot à mot à partir de cette représentation.

---

## Pipeline général

![Worklofw](../assets/Soutenance/workflow.png)

---

## Architecture du modèle de Captioning


### 1. Encodeur (CNN)

Le CNN utilisé est InceptionV3, pré-entraîné sur ImageNet. Il est adapté pour encoder une image en un vecteur de caractéristiques :

```python
base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=(180, 180, 3))

# Couches gelées
for layer in base_model.layers:
    layer.trainable = False

# Pipeline complet
output = base_model.output
output = GlobalAveragePooling2D()(output)       # Réduction spatiale
output = Dense(embedding_dim, activation='relu')(output)  # Embedding

encoder = Model(inputs=base_model.input, outputs=output)
```


## 2. Décodeur (RNN avec LSTM)

Le décodeur prend :
    
- Le vecteur encodé de l’image
- Une séquence de mots (la légende en cours de génération)

Et il génère un mot à chaque étape de la séquence, via un LSTM avec initialisation d’état personnalisée (h et c venant de l’image).

```python
image_features = Input(shape=(embedding_dim,))
caption_input = Input(shape=(max_length,))

embedding = Embedding(input_dim=vocab_size,
                      output_dim=embedding_dim,
                      mask_zero=True)(caption_input)

# Initialisation des états de l'LSTM à partir de l'image
h_initial = Dense(units, activation='relu', name='h_initializer')(image_features)
c_initial = Dense(units, activation='relu', name='c_initializer')(image_features)

lstm_out = LSTM(units, return_sequences=True)(embedding, initial_state=[h_initial, c_initial])
dropout = Dropout(rate=dropout_rate)(lstm_out)
output = Dense(vocab_size, activation='softmax')(dropout)

decoder = Model(inputs=[image_features, caption_input], outputs=output)
```

## Modèle de captioning

```python
# Entrées
image_input = Input(shape=(180, 180, 3), name='image_input')
caption_input = Input(shape=(max_length,), name='caption_input')

# Encodage de l'image
image_features = encoder(image_input)

# Embedding de la séquence texte
embedding = Embedding(input_dim=vocab_size,
                      output_dim=embedding_dim,
                      mask_zero=True)(caption_input)

# États initiaux du LSTM à partir des features image
h_initial = Dense(units, activation='relu')(image_features)
c_initial = Dense(units, activation='relu')(image_features)

# LSTM séquentiel
lstm_out = LSTM(units, return_sequences=True)(embedding, initial_state=[h_initial, c_initial])
dropout = Dropout(rate=dropout_rate)(lstm_out)
output = Dense(vocab_size, activation='softmax')(dropout)

# Modèle combiné
captioning_model = Model(inputs=[image_input, caption_input], outputs=output)
```

# Prétraitement
Images
- Redimensionnement et mise à l’échelle des images.
- Passage dans un CNN pour extraire les features d’encodage.
- (Optionnel) Denoising ou sharpening en amont de l’encodage.

Textes
- Nettoyage des légendes.
- Vectorisation en tokens via un tokenizer personnalisé.
- Padding des séquences.

## Performances du modèle

L'entraînement du modèle est suivi à l’aide de l’objet `history`, qui enregistre la perte (`loss`) pour l'ensemble d'entraînement et de validation.

```python
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.legend()
plt.title("Courbes d'apprentissage")
plt.xlabel("Époques")
plt.ylabel("Loss")
```

Ces courbes permettent de visualiser l’évolution de la fonction de perte au fil des époques, et ainsi d’évaluer la stabilité et la convergence du modèle.

⚠️ Une tentative d'intégration de la métrique BLEU pendant l'entraînement a été faite. Toutefois, le calcul du score BLEU à chaque époque ralentissait significativement l'entraînement, en particulier à cause du volume important de données de validation et du coût de génération des séquences complètes.

Nous avons donc déplacé le calcul du BLEU score à la phase d'inférence, afin d’évaluer les performances du modèle uniquement sur un sous-ensemble d’images tests après entraînement.